# 03 房地产视角的预算缺口分析

从城市房地产开发和销售角度分析 gap_to_gdp 的地区差异和时序特征

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

DATA_CLEAN = r"D:\Project\City_Budget_Analysis\data_clean"
OUTPUT = r"D:\Project\City_Budget_Analysis\output"

df = pd.read_csv(f"{DATA_CLEAN}/city_all_clean.csv")
print(f"数据: {df.shape}")
print(f"列: {df.columns.tolist()}")
df.head()


## 3.1 房地产投资与预算缺口的关系

In [ ]:
# 房地产投资占GDP比重
df['re_invest_to_gdp'] = df['re_invest'] / df['gdp']

# 商品房销售额估算 = 销售面积(万平方米) * 平均价格(元/平方米) / 10000 (转亿元)
df['sale_amount'] = df['sale_area'] * df['re_avg_price'] / 10000
df['sale_to_gdp'] = df['sale_amount'] / df['gdp']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1: 房地产投资/GDP vs gap_to_gdp 散点图
ax = axes[0, 0]
valid = df.dropna(subset=['re_invest_to_gdp', 'gap_to_gdp'])
scatter = ax.scatter(valid['re_invest_to_gdp'], valid['gap_to_gdp'], 
                     c=valid['year'], cmap='viridis', alpha=0.5, s=20)
ax.set_xlabel('房地产投资/GDP')
ax.set_ylabel('gap_to_gdp')
ax.set_title('房地产投资强度 vs 预算缺口')
ax.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax, label='年份')

# 图2: 商品房销售/GDP vs gap_to_gdp
ax = axes[0, 1]
valid2 = df.dropna(subset=['sale_to_gdp', 'gap_to_gdp'])
scatter2 = ax.scatter(valid2['sale_to_gdp'], valid2['gap_to_gdp'], 
                      c=valid2['year'], cmap='viridis', alpha=0.5, s=20)
ax.set_xlabel('商品房销售额/GDP')
ax.set_ylabel('gap_to_gdp')
ax.set_title('商品房销售强度 vs 预算缺口')
ax.grid(alpha=0.3)
plt.colorbar(scatter2, ax=ax, label='年份')

# 图3: 全国平均房地产投资/GDP趋势
ax = axes[1, 0]
avg_re = df.groupby('year')[['re_invest_to_gdp', 'gap_to_gdp']].mean()
ax2 = ax.twinx()
l1, = ax.plot(avg_re.index, avg_re['re_invest_to_gdp'], 'o-', color='#e74c3c', 
              markersize=4, label='房地产投资/GDP')
l2, = ax2.plot(avg_re.index, avg_re['gap_to_gdp'], 's-', color='#3498db', 
               markersize=4, label='gap_to_gdp')
ax.set_xlabel('年份')
ax.set_ylabel('房地产投资/GDP', color='#e74c3c')
ax2.set_ylabel('gap_to_gdp', color='#3498db')
ax.set_title('房地产投资强度与预算缺口趋势')
ax.legend(handles=[l1, l2], loc='upper left')
ax.grid(alpha=0.3)

# 图4: 商品房价格趋势(北上广深)
ax = axes[1, 1]
tier1 = ['北京', '上海', '广州', '深圳']
for city in tier1:
    city_data = df[df['city'] == city]
    ax.plot(city_data['year'], city_data['re_avg_price'], marker='o', markersize=3, label=city)
ax.set_title('北上广深商品房平均销售价格')
ax.set_xlabel('年份')
ax.set_ylabel('元/平方米')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('房地产与预算缺口关系分析', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/real_estate_gap_analysis.png", dpi=150, bbox_inches='tight')
plt.show()


## 3.2 珠三角 vs 长三角 房地产对比

In [ ]:
pearl_river = ['广州', '深圳', '珠海', '佛山', '东莞', '中山', '惠州']
yangtze_river = ['上海', '南京', '杭州', '苏州', '无锡', '宁波', '合肥', '南通']
pearl_available = [c for c in pearl_river if c in df['city'].unique()]
yangtze_available = [c for c in yangtze_river if c in df['city'].unique()]

df_pearl = df[df['city'].isin(pearl_available)]
df_yangtze = df[df['city'].isin(yangtze_available)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1: 房地产投资/GDP对比
ax = axes[0, 0]
pearl_avg = df_pearl.groupby('year')['re_invest_to_gdp'].mean()
yangtze_avg = df_yangtze.groupby('year')['re_invest_to_gdp'].mean()
ax.plot(pearl_avg.index, pearl_avg.values, 'o-', label='珠三角', color='#e74c3c', markersize=4)
ax.plot(yangtze_avg.index, yangtze_avg.values, 's-', label='长三角', color='#3498db', markersize=4)
ax.set_title('房地产投资/GDP')
ax.set_xlabel('年份')
ax.legend()
ax.grid(alpha=0.3)

# 图2: 商品房平均价格对比
ax = axes[0, 1]
pearl_price = df_pearl.groupby('year')['re_avg_price'].mean()
yangtze_price = df_yangtze.groupby('year')['re_avg_price'].mean()
ax.plot(pearl_price.index, pearl_price.values, 'o-', label='珠三角', color='#e74c3c', markersize=4)
ax.plot(yangtze_price.index, yangtze_price.values, 's-', label='长三角', color='#3498db', markersize=4)
ax.set_title('商品房平均销售价格(元/平方米)')
ax.set_xlabel('年份')
ax.legend()
ax.grid(alpha=0.3)

# 图3: 商品房销售面积对比
ax = axes[1, 0]
pearl_area = df_pearl.groupby('year')['sale_area'].mean()
yangtze_area = df_yangtze.groupby('year')['sale_area'].mean()
ax.plot(pearl_area.index, pearl_area.values, 'o-', label='珠三角', color='#e74c3c', markersize=4)
ax.plot(yangtze_area.index, yangtze_area.values, 's-', label='长三角', color='#3498db', markersize=4)
ax.set_title('商品房销售面积(万平方米)')
ax.set_xlabel('年份')
ax.legend()
ax.grid(alpha=0.3)

# 图4: gap_to_gdp 与房地产投资关系(分区域)
ax = axes[1, 1]
ax.scatter(df_pearl['re_invest_to_gdp'], df_pearl['gap_to_gdp'], 
           alpha=0.5, s=20, label='珠三角', color='#e74c3c')
ax.scatter(df_yangtze['re_invest_to_gdp'], df_yangtze['gap_to_gdp'], 
           alpha=0.5, s=20, label='长三角', color='#3498db')
ax.set_xlabel('房地产投资/GDP')
ax.set_ylabel('gap_to_gdp')
ax.set_title('房地产投资强度 vs 预算缺口(分区域)')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('珠三角 vs 长三角 房地产与财政对比', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/pearl_yangtze_real_estate.png", dpi=150, bbox_inches='tight')
plt.show()


## 3.3 房地产市场变化与预算缺口的时序特征

In [ ]:
# 按年份分析房地产市场变化对预算缺口的影响
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 计算相关系数时序
years = sorted(df['year'].unique())
corr_invest = []
corr_price = []
corr_sale = []
for y in years:
    ydata = df[df['year'] == y].dropna(subset=['re_invest_to_gdp', 'gap_to_gdp', 're_avg_price'])
    if len(ydata) > 5:
        corr_invest.append(ydata['re_invest_to_gdp'].corr(ydata['gap_to_gdp']))
        corr_price.append(ydata['re_avg_price'].corr(ydata['gap_to_gdp']))
        corr_sale.append(ydata['sale_area'].corr(ydata['gap_to_gdp']) if 'sale_area' in ydata.columns else np.nan)
    else:
        corr_invest.append(np.nan)
        corr_price.append(np.nan)
        corr_sale.append(np.nan)

# 图1: 相关系数时序
ax = axes[0]
ax.plot(years, corr_invest, 'o-', label='房地产投资/GDP', markersize=4)
ax.plot(years, corr_price, 's-', label='商品房价格', markersize=4)
ax.plot(years, corr_sale, '^-', label='销售面积', markersize=4)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('房地产指标与gap_to_gdp的相关系数')
ax.set_xlabel('年份')
ax.set_ylabel('相关系数')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 图2: 热力图 - 各城市房地产投资/GDP变化
ax = axes[1]
pivot_re = df.pivot_table(index='city', columns='year', values='re_invest_to_gdp')
# 选择部分城市
selected = ['北京', '上海', '广州', '深圳', '杭州', '南京', '成都', '武汉', '重庆', '天津']
selected = [c for c in selected if c in pivot_re.index]
pivot_sel = pivot_re.loc[selected]
im = ax.imshow(pivot_sel.values, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(selected)))
ax.set_yticklabels(selected, fontsize=8)
years_labels = [str(y) for y in pivot_sel.columns]
ax.set_xticks(range(0, len(years_labels), 2))
ax.set_xticklabels([years_labels[i] for i in range(0, len(years_labels), 2)], fontsize=7, rotation=45)
ax.set_title('房地产投资/GDP热力图')
plt.colorbar(im, ax=ax)

# 图3: 房地产投资增速与预算缺口增速
ax = axes[2]
df_sorted = df.sort_values(['city', 'year'])
df_sorted['re_invest_growth'] = df_sorted.groupby('city')['re_invest'].pct_change() * 100
df_sorted['gap_growth'] = df_sorted.groupby('city')['gap'].pct_change() * 100
avg_growth = df_sorted.groupby('year')[['re_invest_growth', 'gap_growth']].mean()
ax.plot(avg_growth.index, avg_growth['re_invest_growth'], 'o-', label='房地产投资增速', 
        color='#e74c3c', markersize=4)
ax.plot(avg_growth.index, avg_growth['gap_growth'], 's-', label='预算缺口增速', 
        color='#3498db', markersize=4)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('房地产投资增速 vs 预算缺口增速(%)')
ax.set_xlabel('年份')
ax.set_ylabel('增速(%)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT}/real_estate_time_series.png", dpi=150, bbox_inches='tight')
plt.show()


## 3.4 综合分析总结

In [ ]:
# 输出关键统计数据
print("=" * 60)
print("综合分析总结")
print("=" * 60)

# 1. 2022年数据概览
df_2022 = df[df['year'] == 2022].dropna(subset=['gap_to_gdp'])
print(f"\n1. 2022年数据概览 ({len(df_2022)} 个城市)")
print(f"   gap_to_gdp 均值: {df_2022['gap_to_gdp'].mean():.4f}")
print(f"   gap_to_gdp 中位数: {df_2022['gap_to_gdp'].median():.4f}")
print(f"   gap_to_gdp 最大: {df_2022.loc[df_2022['gap_to_gdp'].idxmax(), 'city']} ({df_2022['gap_to_gdp'].max():.4f})")
print(f"   gap_to_gdp 最小: {df_2022.loc[df_2022['gap_to_gdp'].idxmin(), 'city']} ({df_2022['gap_to_gdp'].min():.4f})")

# 2. 北上广深对比
print(f"\n2. 2022年北上广深 gap_to_gdp:")
for city in ['北京', '上海', '广州', '深圳']:
    val = df_2022[df_2022['city'] == city]['gap_to_gdp'].values
    if len(val) > 0:
        print(f"   {city}: {val[0]:.4f}")

# 3. 珠三角 vs 长三角
print(f"\n3. 2022年区域对比:")
pearl_2022 = df_2022[df_2022['city'].isin(pearl_available)]['gap_to_gdp'].mean()
yangtze_2022 = df_2022[df_2022['city'].isin(yangtze_available)]['gap_to_gdp'].mean()
print(f"   珠三角平均 gap_to_gdp: {pearl_2022:.4f}")
print(f"   长三角平均 gap_to_gdp: {yangtze_2022:.4f}")

# 4. 房地产相关
print(f"\n4. 2022年房地产相关:")
print(f"   房地产投资/GDP 均值: {df_2022['re_invest_to_gdp'].mean():.4f}")
corr = df_2022[['gap_to_gdp', 're_invest_to_gdp']].corr().iloc[0, 1]
print(f"   房地产投资/GDP 与 gap_to_gdp 相关系数: {corr:.4f}")
